# Ch.11 — Advanced Agentic Patterns

> **Source notes:** `README.md`

Single-pass LLM calls fail on edge cases. This notebook demonstrates four production agentic patterns that trade tokens for reliability:

1. **Reflection** — Draft → Critique → Revise (handle contradictions)
2. **Debate** — Multi-agent consensus (resolve policy ambiguity)
3. **Hierarchical Orchestration** — Planner → Workers → Verifier (complex multi-step tasks)
4. **Tool Selection** — Meta-agent routing with fallback chains (optimize cost/latency)

**Running example:** PizzaBot v2.0 handling contradictory orders, pricing conflicts, and catering requests.

**Setup:** All examples use mock API mode by default (no OpenAI key required). Set `USE_MOCK_API = False` for real LLM calls.

In [ ]:
# TODO: Implement this cell


## 1 · Pattern 1: Reflection Loop — Draft → Critique → Revise

**Scenario:** "Gluten-free, dairy-free pizza with extra cheese" (contradiction)

**How it works:**
```
Step 1: Draft response (single-pass attempt)
Step 2: Self-critique (detect contradiction)
Step 3: Revise response (suggest vegan cheese alternative)
```

**Token economics:**
- Draft: 150 tokens
- Critique: 100 tokens
- Revision: 180 tokens
- **Total: 430 tokens** (2.9× single-pass) → **6× error reduction**

In mock mode, responses are hardcoded to demonstrate the pattern flow.

In [ ]:
# TODO: Implement this cell


## 2 · Pattern 2: Debate — Multi-Agent Consensus

**Scenario:** Pricing conflict — customer has $5 coupon + 10% loyalty + 15% promo active. Which applies?

**How it works:**
```
Agent 1 (generous):  Stack all discounts → $10.75
Agent 2 (strict):    Apply best only → $15.30
Judge (policy check): Queries RAG policy doc → "one discount per order" → Agent 2 wins
```

**Token economics:**
- Agent 1 proposal: 120 tokens
- Agent 2 proposal: 110 tokens
- Agent 1 rebuttal: 130 tokens
- Agent 2 rebuttal: 120 tokens
- Judge + policy lookup: 420 tokens
- **Total: 900 tokens** (6× single-pass) → **8× error reduction**

Debate prevents incorrect discount stacking that costs the business money.

In [ ]:
# TODO: Implement this cell


## 3 · Pattern 3: Hierarchical Orchestration — Planner → Workers → Verifier

**Scenario:** Catering order — 15 pizzas, 3 delivery times (11am, 12pm, 1pm), $200 budget

**How it works:**
```
Planner: Split into 3 batches (5 pizzas each)
Worker 1: Process 11am batch → 5 Margherita = $60
Worker 2: Process 12pm batch → 5 Pepperoni = $70
Worker 3: Process 1pm batch → 3 Veggie + 2 Margherita = $56
Verifier: Check total $186 < $200 ✓, all times feasible ✓
```

**Token economics:**
- Planner: 200 tokens
- 3 workers (parallel): 3 × 150 = 450 tokens
- Verifier: 100 tokens
- **Total: 750 tokens** (5× single-pass) → **15× error reduction**

Hierarchical decomposition prevents single-pass from missing valid solutions.

In [ ]:
# TODO: Implement this cell


## 4 · Pattern 4: Tool Selection with Fallback Chains

**Scenario:** Check pizza inventory with 3 data sources: cache (free, stale), DB (moderate), API (slow, authoritative)

**Strategies:**

1. **Rule-based fallback:** Cache → DB → API → escalate
2. **Cost-optimized:** If time-sensitive → skip cache; else use cache
3. **Meta-agent:** LLM decides which tool based on query semantics

**Token economics:**
- Meta-agent call: 80 tokens
- Tool execution: +0 (cache), +0 (DB), +0 (API)
- **Total: 80-230 tokens** (1.5× single-pass) → **4× error reduction**

Meta-agent routing is worth it when you have >3 tools and complex routing logic.

In [ ]:
# TODO: Implement this cell


## 5 · Error Recovery with Exponential Backoff

**Problem:** Tools fail (timeout, rate limit, transient errors).

**Naive approach:** Single retry → still fails → show error to user ❌

**Production approach:**
```
1. Retry with exponential backoff (1s, 2s, 4s, 8s)
2. Fall back to degraded service (cached/stale data)
3. Escalate to human only after all retries exhausted
```

This pattern is critical for production reliability.

In [ ]:
# TODO: Implement this cell


## 6 · Cost vs Error Reduction Comparison

Now let's compare all four patterns quantitatively:

| Pattern | Tokens | Cost (GPT-4o-mini) | Latency | Error Reduction |
|---------|--------|-------------------|---------|-----------------|
| **Single-pass** | 150 | $0.000023 | 0.5s | baseline (8% error) |
| **Reflection** | 430 | $0.000065 | 1.5s | 6× better (1.2% error) |
| **Debate** | 900 | $0.000135 | 3.0s | 8× better (1.0% error) |
| **Hierarchical** | 750 | $0.000113 | 2.5s | 15× better (0.5% error) |
| **Tool Selection** | 230 | $0.000035 | 0.8s | 4× better (2.0% error) |

**Key insight:** You're trading 2-6× token cost for 4-15× error reduction. For high-stakes production use cases (customer orders, financial transactions), this is a winning trade.

In [ ]:
# TODO: Implement this cell


## 7 · When to Use Each Pattern — Decision Function

How do you decide which pattern to use for a given query?

Here's a decision tree function that recommends the right pattern based on query characteristics:

```
if has_contradiction(query):
    return REFLECTION
elif is_high_stakes(query):
    return DEBATE
elif is_multi_step_complex(query):
    return HIERARCHICAL
elif needs_multiple_tools(query):
    return TOOL_SELECTION
else:
    return SINGLE_PASS
```

Let's test it on real scenarios.

In [ ]:
# TODO: Implement this cell


## 8 · Production Deployment with LangGraph State Machine

In production, these patterns are orchestrated via a **state machine** (LangGraph):

```
┌─────────────┐
│   CLASSIFY  │ ← Determine which pattern to use
└──────┬──────┘
       │
       ▼
┌─────────────┐
│    ROUTE    │ ← Direct to single-pass, reflection, debate, or hierarchical
└──────┬──────┘
       │
       ▼
┌─────────────┐
│   EXECUTE   │ ← Run selected pattern
└──────┬──────┘
       │
       ▼
┌─────────────┐
│   VERIFY    │ ← Validate output, check for hallucinations
└──────┬──────┘
       │
       ▼
   [response]
```

Each pattern becomes a **node** in the graph. LangGraph handles state transitions, retries, and monitoring.

In [ ]:
# TODO: Implement this cell


## 9 · Monitoring and Observability

In production, you need to monitor:

1. **Reflection loop metrics**
   - How many iterations to converge? (avg 2.3, alert if >5)
   - What % of queries need reflection? (target: 15–20%)

2. **Debate metrics**
   - Did agents reach consensus? (target: >90%)
   - Which agent wins most often? (monitor for bias)

3. **Hierarchical orchestration**
   - How many batches per complex order? (avg 3.2)
   - Worker failure rate? (target: <1%)

4. **Tool selection**
   - Which fallback tier is used? (cache: 70%, DB: 25%, API: 5%)
   - How many retries before escalation? (avg 1.8)

Let's simulate a monitoring dashboard.

In [ ]:
# TODO: Implement this cell


## Summary — What You Learned

You now know **four production agentic patterns** that unlock iterative refinement:

1. **Reflection Loop** — Draft → Critique → Revise
   - Use for: Contradictory inputs, edge cases
   - Cost: 2.9× baseline → 6× error reduction

2. **Multi-Agent Debate** — Propose → Challenge → Vote
   - Use for: High-stakes decisions, policy compliance
   - Cost: 6× baseline → 8× error reduction

3. **Hierarchical Orchestration** — Planner → Workers → Verifier
   - Use for: Complex multi-step tasks
   - Cost: 5× baseline → 15× error reduction

4. **Tool Selection with Fallback** — Meta-agent + retry chains
   - Use for: Unreliable tools, cost optimization
   - Cost: 1.5× baseline → 4× error reduction

**Production deployment checklist:**
- ✓ Route queries through pattern classifier
- ✓ Orchestrate with LangGraph state machine
- ✓ Monitor convergence, consensus, tool usage
- ✓ Set budget alerts (cost per conversation)
- ✓ A/B test single-pass vs reflection (50/50 split)

**Next:** [Azure Supplement](notebook_supplement.ipynb) — Deploy these patterns to production with Azure Container Instances, monitor with Application Insights, and run A/B tests.